In [ ]:
!pip install boto3 -q
import os, boto3, json, time

# ===========================================================
# STEP 1: Simulate model checkpoints
# ===========================================================

# Folder to store checkpoints
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Simulate saving model weights (dummy files)
for epoch in range(1, 4):
    checkpoint_data = {
        "epoch": epoch,
        "model_state_dict": f"dummy_weights_epoch_{epoch}",
        "optimizer_state_dict": f"dummy_optimizer_epoch_{epoch}",
        "timestamp": time.ctime()
    }
    file_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.json")
    with open(file_path, "w") as f:
        json.dump(checkpoint_data, f, indent=2)
    print(f"💾 Saved: {file_path}")

# ===========================================================
# STEP 2: Configure AWS Credentials
# ===========================================================
os.environ["AWS_ACCESS_KEY_ID"] = "AGFJ657HGJGCT6BHJ96U"
os.environ["AWS_SECRET_ACCESS_KEY"] = "4CGHCMVGH887T66UGKY/CGHMHVGK8z5Wl"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

# ===========================================================
# STEP 3: Upload Checkpoints to S3
# ===========================================================
s3 = boto3.client("s3")
bucket_name = "bert-lora-bucket"   # 🔹 replace with your actual bucket
s3_folder = "model-checkpoints/"

# Upload all files
for file_name in os.listdir(checkpoint_dir):
    local_path = os.path.join(checkpoint_dir, file_name)
    s3_path = os.path.join(s3_folder, file_name)
    s3.upload_file(local_path, bucket_name, s3_path)
    print(f"✅ Uploaded {file_name} → s3://{bucket_name}/{s3_path}")

print(f"\n🎉 All checkpoints uploaded successfully to s3://{bucket_name}/{s3_folder}")

# ===========================================================
# STEP 4: Verify Upload (List files)
# ===========================================================
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_folder)
print("\n📂 Files in S3:")
for obj in response.get("Contents", []):
    print(f" - {obj['Key']} ({obj['Size']} bytes)")


💾 Saved: checkpoints/checkpoint_epoch_1.json
💾 Saved: checkpoints/checkpoint_epoch_2.json
💾 Saved: checkpoints/checkpoint_epoch_3.json
✅ Uploaded checkpoint_epoch_1.json → s3://my-bert-lora-bucket/model-checkpoints/checkpoint_epoch_1.json
✅ Uploaded checkpoint_epoch_2.json → s3://my-bert-lora-bucket/model-checkpoints/checkpoint_epoch_2.json
✅ Uploaded checkpoint_epoch_3.json → s3://my-bert-lora-bucket/model-checkpoints/checkpoint_epoch_3.json

🎉 All checkpoints uploaded successfully to s3://my-bert-lora-bucket/model-checkpoints/

📂 Files in S3:
 - model-checkpoints/checkpoint_epoch_1.json (159 bytes)
 - model-checkpoints/checkpoint_epoch_2.json (159 bytes)
 - model-checkpoints/checkpoint_epoch_3.json (159 bytes)
